# Category extraction & trusted sources playground

Step-by-step **category extraction** uses local **Ollama**. **Trusted sources** uses **Groq** (`GROQ_API_KEY` in `.env`).

In [ ]:
import logging
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "settings.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.INFO, force=True)


In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

from config.settings import LOCAL_MODEL_SMALL, OLLAMA_BASE_URL
from prompts.category_extraction import (
    CATEGORY_EXTRACTION_SYSTEM,
    CATEGORY_EXTRACTION_USER_TEMPLATE,
)
from schemas.category_extraction import (
    CategoryComplete,
    CategoryNeedsClarification,
    CategoryNoShoppingIntent,
)


In [ ]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "{system_prompt}"),
        ("human", "{user_prompt}"),
    ]
)

json_parser = JsonOutputParser()


### 1. Construct ChatOllama

In [ ]:
llm = ChatOllama(
    model=LOCAL_MODEL_SMALL,
    base_url=OLLAMA_BASE_URL.rstrip("/"),
    temperature=0,
)

### 2. Build user message


In [ ]:
# --- edit these ---
QUERY = "I'm planning to buy a journal to write down my travel stories"

# Optional: uncomment and fill for a follow-up turn (same QUERY as round 1).
# clarification_context = ""
clarification_context = """
Q: Will you mainly use it at a desk or carry it while travelling?
A: I'll mostly use it at a desk but will carry it with me when I travel
Q: Are you looking for a physical paper notebook or a digital journaling app?
A: I'm looking for a physical paper notebook
Q: Do you have a preferred size in mind?
A: I'm open to any size
"""

In [ ]:
user_prompt = CATEGORY_EXTRACTION_USER_TEMPLATE.format(
    user_query=QUERY.strip(),
    clarification_context=clarification_context.strip(),
)
print(user_prompt)


Shopper request: I'm planning to buy a journal to write down my travel stories
Q: Will you mainly use it at a desk or carry it while travelling?
A: I'll mostly use it at a desk but will carry it with me when I travel
Q: Are you looking for a physical paper notebook or a digital journaling app?
A: I'm looking for a physical paper notebook
Q: Do you have a preferred size in mind?
A: I'm open to any size



In [ ]:
messages = prompt_template.format_messages(
    system_prompt=CATEGORY_EXTRACTION_SYSTEM,
    user_prompt=user_prompt,
)

messages

[SystemMessage(content='You are the structured parsing step for a shopping assistant. Given a shopper’s free‑text request (and any prior clarification answers), you must do **exactly one** of:\n- (**A**) Return **`complete`** with `category` plus search‑helpful attributes when the product kind is clear enough.\n- (**B**) Return **`needs_clarification`** with focused questions when important ambiguities remain.\n- (**C**) Return **`no_shopping_intent`** when there is **no plausible product‑shopping intent** (gibberish, empty pleasantries, unrelated topics, or text that cannot reasonably be interpreted as looking to buy/research something to purchase).\n\n## Shopper message layout\nThe message begins with **`Shopper request:`** and the **original** query. **Any following lines** are the clarification block: when **absent or blank**, this is the first pass; when **present**, they carry the **prior clarification exchange** (questions asked and shopper answers) formatted by the caller. Trea

In [ ]:
response = llm.invoke(messages)
response


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


AIMessage(content='{"status":"complete","category":"notebook","is_digital":false,"size":"any"}', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-05-16T19:03:00.285154173Z', 'done': True, 'done_reason': 'stop', 'total_duration': 93814696356, 'load_duration': 27661385918, 'prompt_eval_count': 1065, 'prompt_eval_duration': 63587961310, 'eval_count': 21, 'eval_duration': 2451660259, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019e322a-1737-7f51-92df-0d99c6b494a1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1065, 'output_tokens': 21, 'total_tokens': 1086})

In [ ]:
raw = response.content
type(raw), raw


(str,
 '{"status":"complete","category":"notebook","is_digital":false,"size":"any"}')

### 3. Normalize to one string


In [ ]:
if isinstance(raw, str):
    reply_text = raw
elif isinstance(raw, list):
    reply_text = "".join(
        b if isinstance(b, str) else (b.get("text", "") if isinstance(b, dict) else "")
        for b in raw
    )
else:
    raise TypeError(f"Unexpected content type: {type(raw)}")

reply_text = reply_text.strip()
reply_text


'{"status":"complete","category":"notebook","is_digital":false,"size":"any"}'

### 4. Parse JSON (JsonOutputParser)


In [ ]:
payload = json_parser.parse(reply_text)
payload


{'status': 'complete',
 'category': 'notebook',
 'is_digital': False,
 'size': 'any'}

### 5. Validate into Pydantic models


In [ ]:
if not isinstance(payload, dict):
    raise TypeError("Expected JSON object")

status = payload.get("status")

if status == "needs_clarification":
    result = CategoryNeedsClarification.model_validate(payload)
elif status == "complete":
    result = CategoryComplete.model_validate(payload)
elif status == "no_shopping_intent":
    result = CategoryNoShoppingIntent.model_validate(payload)
else:
    raise ValueError(f"Unknown status: {status!r}")

type(result), result.model_dump()


(schemas.category_extraction.CategoryComplete,
 {'status': 'complete',
  'category': 'notebook',
  'is_digital': False,
  'size': 'any'})

## Trusted sources (Groq)

Production passes **category only** (no full shopper query) so domain picks stay tied to the review vertical.

Requires **`GROQ_API_KEY`** in `.env` (loaded via `config.settings`).

In [ ]:
from langchain_groq import ChatGroq

from config.settings import GROQ_API_KEY, GROQ_MODEL_SMALL
from prompts.trusted_sources import TRUSTED_SOURCES_SYSTEM, TRUSTED_SOURCES_USER_TEMPLATE
from schemas.trusted_sources import TrustedSourcesResponse

### Trusted sources — quick path (`fetch_trusted_sources_domains`)

In [ ]:
from services.trusted_sources import fetch_trusted_sources_domains

CATEGORY = "laptop"  # edit — use coarse vertical from category extraction

if not GROQ_API_KEY:
    raise RuntimeError("Set GROQ_API_KEY in .env (project root)")

domains = fetch_trusted_sources_domains(CATEGORY)
domains

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


['cnet.com',
 'pcmag.com',
 'tomshardware.com',
 'wired.com',
 'theverge.com',
 'rtings.com',
 'laptopmag.com',
 'digitaltrends.com',
 'techradar.com',
 'pcworld.com']

### Trusted sources — step-by-step (same stack as `services/trusted_sources.py`)

Requires **`CATEGORY`** from the quick-path cell above (or set `CATEGORY = "..."` yourself).

Reuses **`prompt_template`** and **`json_parser`** from the category section above.

In [ ]:
user_prompt_ts = TRUSTED_SOURCES_USER_TEMPLATE.format(category=CATEGORY.strip())
print(user_prompt_ts)

messages_ts = prompt_template.format_messages(
    system_prompt=TRUSTED_SOURCES_SYSTEM,
    user_prompt=user_prompt_ts,
)
messages_ts

Product category: laptop



[SystemMessage(content='You are the research librarian for a shopping assistant. Your job is to propose **trusted editorial sources** (publishers and labs) that help a shopper understand a category and compare products.\n\n## What counts as “trusted editorial”\nPrefer sources that routinely publish **independent product evaluation**: methodology, criteria, hands-on testing or structured benchmarking, updates when models change, and clear separation between editorial recommendations and commerce.\n\nGood signals include:\n- Established editorial brands, nonprofit labs, specialized magazines, or respected enthusiast publications with editorial standards.\n- Content types like **buying guides**, **face‑off comparisons**, **best picks**, **long‑term tests**, and **explainer** articles grounded in experience or measurement.\n\n## What to avoid\n- Manufacturer / brand official sites as “review” sources.\n- Retailers whose pages are mostly listings, promotions, or thin SEO summaries without s

### Invoke ChatGroq

In [ ]:
llm_groq = ChatGroq(model=GROQ_MODEL_SMALL, api_key=GROQ_API_KEY, temperature=0)

response_ts = llm_groq.invoke(messages_ts)
response_ts

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


AIMessage(content='["cnet.com","pcmag.com","tomshardware.com","wired.com","theverge.com","rtings.com","laptopmag.com","digitaltrends.com","techradar.com","pcworld.com"]', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 554, 'total_tokens': 601, 'completion_time': 0.2935086, 'completion_tokens_details': None, 'prompt_time': 0.039442061, 'prompt_tokens_details': None, 'queue_time': 0.034577655, 'total_time': 0.332950661}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e331c-2b65-7882-9762-75424851335a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 554, 'output_tokens': 47, 'total_tokens': 601})

### Normalize content → parse JSON → `TrustedSourcesResponse`

In [ ]:
raw_ts = response_ts.content
if isinstance(raw_ts, str):
    reply_ts = raw_ts.strip()
elif isinstance(raw_ts, list):
    reply_ts = "".join(
        b if isinstance(b, str) else (b.get("text", "") if isinstance(b, dict) else "")
        for b in raw_ts
    ).strip()
else:
    raise TypeError(f"Unexpected content type: {type(raw_ts)}")

reply_ts

'["cnet.com","pcmag.com","tomshardware.com","wired.com","theverge.com","rtings.com","laptopmag.com","digitaltrends.com","techradar.com","pcworld.com"]'

In [ ]:
payload_ts = json_parser.parse(reply_ts)
payload_ts

['cnet.com',
 'pcmag.com',
 'tomshardware.com',
 'wired.com',
 'theverge.com',
 'rtings.com',
 'laptopmag.com',
 'digitaltrends.com',
 'techradar.com',
 'pcworld.com']

In [ ]:
trusted = TrustedSourcesResponse.model_validate(payload_ts)
trusted.model_dump()

{'domains': ['cnet.com',
  'pcmag.com',
  'tomshardware.com',
  'wired.com',
  'theverge.com',
  'rtings.com',
  'laptopmag.com',
  'digitaltrends.com',
  'techradar.com',
  'pcworld.com']}